In [23]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels

import pandas as pd
import numpy as np
import pyblp 
import statsmodels.api as sm_api
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   -------------------------------------

'1.1.2'

In [16]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)


In [17]:
# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares',
    'cost_shifter' : 'demand_instruments0',
}, inplace=True)


In [5]:
################ Nested Logit Model with Charging
def solve_nl(df):
    groups = df.groupby(['market_ids', 'nesting_ids'])
    df['demand_instruments1'] = groups['shares'].transform(np.size)
    nl_formulation = pyblp.Formulation('0 + prices + is_electric*log_charging_IV')
    problem = pyblp.Problem(nl_formulation, df)
    return problem.solve(rho=0.66)

# Solve the nested logit model
results = solve_nl(df)

# Display results
print(results)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +9.2E-22     +8.3E-09     +2.7E+04     0         +1.8E+13          +1.3E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective 
   Time      Converged   Iterations   Evaluations
-----------  ---------  ------------  -----------
 00:01:46       Yes          4            13     

Rho Estimates (Robust SEs in Parentheses):
All Groups
----------
 +6.6E-01 
(+8.6E-03)

Beta Estimates (Robust SEs in Parentheses):
  prices    is_electric  log_charging_IV  is_electric*log_charging_IV
----------  -----------  ---------------  ---------------------------
 -8.1E-02    -1.2E+01       -5.4E-01               +1.1E+00          
(+1.2E-03)  (+1.0E-01)     (+6.1E-03)             (+1.1E-02)

In [ ]:
################ Compute for own- and cross-price elasticities
elasticities = results.compute_elasticities()

In [ ]:
################ Supply side model
# Extract by-market observations
province_year_df = (
    df
    .drop_duplicates(subset=['province', 'year'])
    [['province', 'year', 'charging_stations_stock', 'EV_stock', 'road_fuel_IV']]
    .reset_index(drop=True)
)

print(province_year_df.shape)
# Create a time trend variable
province_year_df['time_trend'] = province_year_df['year'].astype(int) - province_year_df['year'].astype(int).min() + 1

# Add time_trend to your formula
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV] + C(province)+ time_trend'

model = IV2SLS.from_formula(formula, data=province_year_df).fit(
    cov_type='clustered',
    clusters=province_year_df['province'],
)
print(model.summary)

(155, 5)
                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9697
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9620
No. Observations:                           155   F-statistic:                -7.991e+17
Date:                          Tue, Jul 22 2025   P-value (F-stat)                1.0000
Time:                                  17:07:09   Distribution:                 chi2(32)
Cov. Estimator:                       clustered                                         
                                                                                        
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
C(provinc

In [84]:
# Run first stage regression
first_stage = IV2SLS.from_formula(
    'log(charging_stations_stock) ~ road_fuel_IV + C(province) + C(year)',
    data=province_year_df
).fit(cov_type='clustered', clusters=province_year_df['province'])
print(first_stage.summary)

                                 OLS Estimation Summary                                 
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9759
Estimator:                                  OLS   Adj. R-squared:                 0.9688
No. Observations:                           155   F-statistic:                -1.037e+19
Date:                          Tue, Jul 22 2025   P-value (F-stat)                1.0000
Time:                                  17:52:27   Distribution:                 chi2(35)
Cov. Estimator:                       clustered                                         
                                                                                        
                                    Parameter Estimates                                    
                         Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------
Intercept   